In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig
import torch

model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"
# model_id = "Qwen/Qwen2.5-Coder-14B-Instruct"


device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=QuantoConfig(weights="int8"),
    torch_dtype=torch.float16
    ).to(device)
max_new_tokens = 512
print(
    model.get_memory_footprint(
        return_buffers=True
    ) / 1024**3,
    "GB")

/Users/rojankarki/Projects/watermark-llm-code-quality/watermark-llm-codebase/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:16<00:00, 20.57it/s]


14.185193091630936 GB


In [2]:
print(model.dtype)                          # e.g. torch.float16
print(next(model.parameters()).dtype)       # cross-check, same result
print(model.get_memory_footprint() / 1e9, "GB")

torch.float16
torch.float16
15.231235104 GB


# Check Prepare Query and Response using a sample from dataset

In [3]:
def prepare_query(data, language = "python"):
    language = "python" 
    hint = data["hints"].get(language, "").strip()
    hint_line = f"Hint: {hint}\n" if hint else ""

    query = f"""Write a {language} code for the following task description: {data['prompt_description']}
{hint_line} """
    return query

In [21]:
import json

def read_json(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        dataset = json.load(f)
    return dataset

def write_json(filename, json_data):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(json_data, f, indent=2, ensure_ascii=False)

In [4]:
data = {
      "task_number": 11,
      "prompt_title": "Growth of a Population",
      "prompt_description": "In a small town the population is p0 = 1000 at the beginning of a year. The population regularly increases by 2 percent per year and moreover 50 new inhabitants per year come to live in the town. How many years does the town need to see its population greater than or equal to p = 1200 inhabitants?\n\nAt the end of the first year there will be: \n1000 + 1000 * 0.02 + 50 => 1070 inhabitants\n\nAt the end of the 2nd year there will be: \n1070 + 1070 * 0.02 + 50 => 1141 inhabitants (** number of inhabitants is an integer **)\n\nAt the end of the 3rd year there will be:\n1141 + 1141 * 0.02 + 50 => 1213\n\nIt will need 3 entire years.\nMore generally given parameters:\n\np0, percent, aug (inhabitants coming or leaving each year), p (population to equal or surpass)\n\nthe function nb_year should return n number of entire years needed to get a population greater or equal to p.\n\naug is an integer, percent a positive or null floating number, p0 and p are positive integers (> 0)\n\nExamples:\nnb_year(1500, 5, 100, 5000) -> 15\nnb_year(1500000, 2.5, 10000, 2000000) -> 10\nNote:\nDon't forget to convert the percent parameter as a percentage in the body of your function: if the parameter percent is 2 you have to convert it to 0.02.\n\nThere are no fractions of people. At the end of each year, the population count is an integer: 252.8 people round down to 252 persons.",
      "hints": {
        "java": "",
        "c": "",
        "cpp": "",
        "python": ""
      },
      "solutions": {
        "java": "",
        "c": "",
        "cpp": "",
        "python": ""
      },
      "source": "https://www.codewars.com/dashboard",
      "tags": ["ALGORITHMS"],
      "comments": ""
    }
query = prepare_query(data)
print(query)

Write a python code for the following task description: In a small town the population is p0 = 1000 at the beginning of a year. The population regularly increases by 2 percent per year and moreover 50 new inhabitants per year come to live in the town. How many years does the town need to see its population greater than or equal to p = 1200 inhabitants?

At the end of the first year there will be: 
1000 + 1000 * 0.02 + 50 => 1070 inhabitants

At the end of the 2nd year there will be: 
1070 + 1070 * 0.02 + 50 => 1141 inhabitants (** number of inhabitants is an integer **)

At the end of the 3rd year there will be:
1141 + 1141 * 0.02 + 50 => 1213

It will need 3 entire years.
More generally given parameters:

p0, percent, aug (inhabitants coming or leaving each year), p (population to equal or surpass)

the function nb_year should return n number of entire years needed to get a population greater or equal to p.

aug is an integer, percent a positive or null floating number, p0 and p are p

# MBPP


In [3]:
import json
dataset = []
with open("/Users/rojankarki/Projects/watermark-llm-code-quality/watermark-llm-test-dataset/dataset/mbpp/mbpp.jsonl", "r") as f:
    for line in f:
        dataset.append(json.loads(line))

print(len(dataset))
print(json.dumps(dataset[0], indent=2))
print(json.dumps(dataset[1], indent=2))

974
{
  "text": "Write a function to find the minimum cost path to reach (m, n) from (0, 0) for the given cost matrix cost[][] and a position (m, n) in cost[][].",
  "code": "R = 3\r\nC = 3\r\ndef min_cost(cost, m, n): \r\n\ttc = [[0 for x in range(C)] for x in range(R)] \r\n\ttc[0][0] = cost[0][0] \r\n\tfor i in range(1, m+1): \r\n\t\ttc[i][0] = tc[i-1][0] + cost[i][0] \r\n\tfor j in range(1, n+1): \r\n\t\ttc[0][j] = tc[0][j-1] + cost[0][j] \r\n\tfor i in range(1, m+1): \r\n\t\tfor j in range(1, n+1): \r\n\t\t\ttc[i][j] = min(tc[i-1][j-1], tc[i-1][j], tc[i][j-1]) + cost[i][j] \r\n\treturn tc[m][n]",
  "task_id": 1,
  "test_setup_code": "",
  "test_list": [
    "assert min_cost([[1, 2, 3], [4, 8, 2], [1, 5, 3]], 2, 2) == 8",
    "assert min_cost([[2, 3, 4], [5, 9, 3], [2, 6, 4]], 2, 2) == 12",
    "assert min_cost([[3, 4, 5], [6, 10, 4], [3, 7, 5]], 2, 2) == 16"
  ],
  "challenge_test_list": []
}
{
  "text": "Write a function to find the similar elements from the given two tuple lists.

In [14]:
def prepare_mbpp_query(data, sample, language = "python"):
    requirement = f"""
Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.
"""
    one_shot_example = f"""Write a {language} code for the following task description: {sample['text']}
{requirement}
### EXAMPLE
Test cases: {sample['test_list']}
Output: ```{language}
{sample['code']}
```"""
    query = f"""{one_shot_example}
### TARGET
Now, write a {language} code for the following task description: {data['text']}
{requirement}
Test cases: {data['test_list']}
Output: 
"""
    return query

In [15]:
one_shot_example = sample=dataset[1]
query = prepare_mbpp_query(dataset[888], one_shot_example )
print(query)

Write a python code for the following task description: Write a function to find the similar elements from the given two tuple lists.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

### EXAMPLE
Test cases: ['assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)', 'assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)', 'assert similar_elements((11, 12, 14, 13),(17, 15, 14, 13)) == (13, 14)']
Output: ```python
def similar_elements(test_tup1, test_tup2):
  res = tuple(set(test_tup1) & set(test_tup2))
  return (res) 
```
### TARGET
Now, write a python code for the following task description: Write a function to reverse each list in a given list of lists.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

Test cases: ['assert reverse_list_lists([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]])==[[4, 3, 2, 1], [8, 7, 6, 5], [12, 11, 10, 9], [16, 15, 14, 13]]', '

In [16]:
messages = [
    {"role": "user", "content": query}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

# inputs = tokenizer(query, return_tensors="pt", add_special_tokens = True).to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.1,
    )


# response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
response = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]

print(response)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a python code for the following task description: Write a function to find the similar elements from the given two tuple lists.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

### EXAMPLE
Test cases: ['assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)', 'assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)', 'assert similar_elements((11, 12, 14, 13),(17, 15, 14, 13)) == (13, 14)']
Output: ```python
def similar_elements(test_tup1, test_tup2):
  res = tuple(set(test_tup1) & set(test_tup2))
  return (res) 
```
### TARGET
Now, write a python code for the following task description: Write a function to reverse each list in a given list of lists.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

Test cases: ['assert reverse_list_lists([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 1

# MARKLLM Framework

In [17]:
from watermark.auto_watermark import AutoWatermark
from utils.transformers_config import TransformersConfig

# Transformers config
transformers_config = TransformersConfig(model=model,
                                         tokenizer=tokenizer,
                                         device=device,
                                         max_new_tokens=max_new_tokens,
                                         min_length=230,
                                         do_sample=True,
                                         no_repeat_ngram_size=4,
                                        #  temperature=0.1,
                                         )


In [18]:
# Load watermark algorithm
myWatermark = AutoWatermark.load('SynthID', 
                                 algorithm_config='config/SynthID.json',
                                 transformers_config=transformers_config)

In [19]:
result = []
one_shot_example = sample=dataset[1]
for i, data in enumerate(dataset[10:30]):
    print(f"Processing index: {i}")
    query = prepare_mbpp_query(data, sample)
    messages = [
        {"role": "user", "content": query}
    ]

    query = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    watermarked_text = myWatermark.generate_watermarked_text(query)
    unwatermarked_text = myWatermark.generate_unwatermarked_text(query)
    detect_result_watermarked_text = myWatermark.detect_watermark(watermarked_text)
    detect_result_unwatermarked_text = myWatermark.detect_watermark(unwatermarked_text)
    result.append({
        **data,
        "outputs": {
            "watermarked": {
                "content": watermarked_text,
                **detect_result_watermarked_text,
            },
            "unwatermarked":{
                "content": unwatermarked_text,
                **detect_result_unwatermarked_text,
            }
        }
    })

print(json.dumps(result, indent=2))

Processing index: 0
Processing index: 1
Processing index: 2
Processing index: 3
Processing index: 4
Processing index: 5
Processing index: 6
Processing index: 7
Processing index: 8
Processing index: 9
Processing index: 10
Processing index: 11
Processing index: 12
Processing index: 13
Processing index: 14
Processing index: 15
Processing index: 16
Processing index: 17
Processing index: 18
Processing index: 19
[
  {
    "text": "Write a python function to remove first and last occurrence of a given character from the string.",
    "code": "def remove_Occ(s,ch): \r\n    for i in range(len(s)): \r\n        if (s[i] == ch): \r\n            s = s[0 : i] + s[i + 1:] \r\n            break\r\n    for i in range(len(s) - 1,-1,-1):  \r\n        if (s[i] == ch): \r\n            s = s[0 : i] + s[i + 1:] \r\n            break\r\n    return s ",
    "task_id": 11,
    "test_setup_code": "",
    "test_list": [
      "assert remove_Occ(\"hello\",\"l\") == \"heo\"",
      "assert remove_Occ(\"abcda\",\"a\

In [22]:
write_json('../result-gwen-prompt-refined-10-30.json', result)

In [21]:
for i in range(2):
    result = []
    for data in dataset:
        query = prepare_query(data)
        messages = [
            {"role": "user", "content": query}
        ]

        query = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        watermarked_text = myWatermark.generate_watermarked_text(query)
        unwatermarked_text = myWatermark.generate_unwatermarked_text(query)
        detect_result_watermarked_text = myWatermark.detect_watermark(watermarked_text)
        detect_result_unwatermarked_text = myWatermark.detect_watermark(unwatermarked_text)
        result.append({
            **data,
            "outputs": {
                "watermarked": {
                    "content": watermarked_text,
                    **detect_result_watermarked_text,
                },
                "unwatermarked":{
                    "content": unwatermarked_text,
                    **detect_result_unwatermarked_text,
                }
            }
        })
    filename = f"result-{i}.json"
    write_json(filename, result)


KeyboardInterrupt: 

In [29]:
print(result[19]['outputs']['watermarked']['content'])

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a python code for the following task description: Write a function to find the similar elements from the given two tuple lists.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

### EXAMPLE
Test cases: ['assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)', 'assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)', 'assert similar_elements((11, 12, 14, 13),(17, 15, 14, 13)) == (13, 14)']
Output: ```python
def similar_elements(test_tup1, test_tup2):
  res = tuple(set(test_tup1) & set(test_tup2))
  return (res) 
```
### TARGET
Now, write a python code for the following task description: Write a python function to count all the substrings starting and ending with same characters.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

Test cases: ['assert count_Substring_With_Equal_Ends("abc") ==

In [30]:
print(result[17]['outputs']['unwatermarked']['content'])

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a python code for the following task description: Write a function to find the similar elements from the given two tuple lists.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

### EXAMPLE
Test cases: ['assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)', 'assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)', 'assert similar_elements((11, 12, 14, 13),(17, 15, 14, 13)) == (13, 14)']
Output: ```python
def similar_elements(test_tup1, test_tup2):
  res = tuple(set(test_tup1) & set(test_tup2))
  return (res) 
```
### TARGET
Now, write a python code for the following task description: Write a python function to find binomial co-efficient.

Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.

Test cases: ['assert binomial_Coeff(5,2) == 10', 'assert binomial_Coeff(4,3) == 4', 'assert binomial_

# MISC

In [29]:
import re

def extract_code_block(output: str, language: str = "python") -> str:
    pattern = rf"```{language}\s*\n(.*?)```"
    match = re.search(pattern, output, re.DOTALL)
    if match:
        return match.group(1).strip()
    return ""

In [37]:
watermarked_text_code = extract_code_block(result[0]['outputs']['watermarked']['content'])
print(watermarked_text_code)

import os
import threading
from queue import Queue

def process_file(file_path, output_queue):
    with open겥


In [31]:
unwatermarked_text_code = extract_code_block(unwatermarked_text)
print(unwatermarked_text_code)

def reverse_five_or_more(s):
    return ' '.join(word[::-1] if len(word) >= 5 else word for word in s.split())

# Test cases
print(reverse_five_or_more("Hey fellow warriors"))  # Output: "Hey wollef sroirraw"
print(reverse_five_or_more("This is a test"))       # Output: "This is a test"
print(reverse_five_or_more("This is another test")) # Output: "This is rehtona test"


In [33]:
def strip_prompt(output: str, query: str) -> str:
    if output.startswith(query):
        return output[len(query):].strip()
    # fallback: find query anywhere and take everything after it
    idx = output.find(query)
    if idx != -1:
        return output[idx + len(query):].strip()
    return output.strip()

unwatermarked = strip_prompt(unwatermarked_text, query)
print(unwatermarked)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a python code for the following task description: Write a function that takes in a string of one or more words, and returns the same string, but with all words that have five or more letters reversed (Just like the name of this Kata). Strings passed in will consist of only letters and spaces. Spaces will be included only when more than one word is present.

Examples:

"Hey fellow warriors"  --> "Hey wollef sroirraw" 
"This is a test        --> "This is a test" 
"This is another test" --> "This is rehtona test"
Output only the raw python code — no explanations, no markdown code fences, no preamble or postamble. The code must be complete and directly executable when saved to a file 
assistant
```python
def reverse_five_or_more(s):
    return ' '.join(word[::-1] if len(word) >= 5 else word for word in s.split())

# Test cases
print(reverse_five_or_more("Hey fellow warriors"))  # Output: "Hey wollef sroi

# VISUALIZATION

In [34]:
from visualize.font_settings import FontSettings
from visualize.visualizer import DiscreteVisualizer
from visualize.legend_settings import DiscreteLegendSettings
from visualize.page_layout_settings import PageLayoutSettings
from visualize.color_scheme import ColorSchemeForDiscreteVisualization

In [35]:
watermarked_data = myWatermark.get_data_for_visualization(watermarked_text)
unwatermarked_data = myWatermark.get_data_for_visualization(unwatermarked_text)

# Init visualizer
visualizer = DiscreteVisualizer(color_scheme=ColorSchemeForDiscreteVisualization(),
                                font_settings=FontSettings(), 
                                page_layout_settings=PageLayoutSettings(),
                                legend_settings=DiscreteLegendSettings())
# Visualize
watermarked_img = visualizer.visualize(data=watermarked_data, 
                                       show_text=True, 
                                       visualize_weight=True, 
                                       display_legend=True)

unwatermarked_img = visualizer.visualize(data=unwatermarked_data,
                                         show_text=True, 
                                         visualize_weight=True, 
                                         display_legend=True)

In [36]:
from PIL import Image

def side_by_side(img1: Image.Image, img2: Image.Image) -> Image.Image:
    w = img1.width + img2.width
    h = max(img1.height, img2.height)
    combined = Image.new("RGB", (w, h), (255, 255, 255))
    combined.paste(img1, (0, 0))
    combined.paste(img2, (img1.width, 0))
    return combined

combined_img = side_by_side(watermarked_img, unwatermarked_img)
combined_img.show()  # opens in default image viewer